In [8]:
import numpy as np
import pandas as pd
import warnings

from itertools import combinations
from math import comb
from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

### File Load

In [9]:
df = pd.read_csv(r"../../../data/processed/data_vif.csv")
target = "Risk_Label"

n_total = len(df)
n_train = int(n_total * 0.5)
n_valid = int(n_total * 0.3)
n_test = n_total - n_train - n_valid

#Date 컬럼 제거
df.drop("Date", axis=1, inplace=True)

# Risk_Label 매핑 (High Risk=1, Low Risk=0)
df["Risk_Label"] = df["Risk_Label"].map({"Low Risk": 0, "High Risk": 1})

#train/valid/test 분할 5/3/2
train_df = df.iloc[:n_train].reset_index(drop=True)
valid_df = df.iloc[n_train : n_train + n_valid].reset_index(drop=True)
test_df = df.iloc[n_train + n_valid :].reset_index(drop=True)

print(f"total rows: {n_total}, train: {len(train_df)}, valid: {len(valid_df)}, test: {len(test_df)}")

# Split features/target
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid = valid_df.drop(columns=[target])
y_valid = valid_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

print("train shape:", X_train.shape, y_train.shape)
print("valid shape:", X_valid.shape, y_valid.shape)
print("test shape:", X_test.shape, y_test.shape)

total rows: 4108, train: 2054, valid: 1232, test: 822
train shape: (2054, 28) (2054,)
valid shape: (1232, 28) (1232,)
test shape: (822, 28) (822,)


## Min-Max Scaling

In [10]:
scaler = MinMaxScaler().set_output(transform="pandas")

# train에만 fit하고 valid/test에는 같은 scaler를 적용
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

print('train/valid/test:', len(X_train), len(X_valid), len(X_test))
print('y_train class:', y_train.value_counts().to_dict())
print('y_valid class:', y_valid.value_counts().to_dict())
print('y_test class:', y_test.value_counts().to_dict())

train/valid/test: 2054 1232 822
y_train class: {0: 1859, 1: 195}
y_valid class: {0: 1070, 1: 162}
y_test class: {0: 730, 1: 92}


## 파라미터, 지표

In [11]:
# ============================================================
# 1. Metric 함수
# ============================================================

def gmean_score(y_true, y_pred):
    """
    binary label 기준:
    0 = Low Risk
    1 = High Risk

    G-Mean = sqrt(Recall * Specificity)
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    return np.sqrt(recall * specificity)


def classification_metrics(y_true, y_pred, y_prob=None):
    """
    각종 성능지표 계산.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall,
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "gmean": np.sqrt(recall * specificity),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

    if y_prob is not None:
        try:
            metrics["auc"] = roc_auc_score(y_true, y_prob)
        except Exception:
            metrics["auc"] = np.nan
    else:
        metrics["auc"] = np.nan

    return metrics


def find_best_threshold(y_true, y_prob, threshold_grid=None):
    """
    valid set에서 G-Mean이 최대가 되는 threshold 탐색.
    """
    if threshold_grid is None:
        threshold_grid = [0.05, 0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]

    best_threshold = 0.5
    best_metrics = None
    best_pred = None

    for threshold in threshold_grid:
        y_pred = (y_prob >= threshold).astype(int)
        metrics = classification_metrics(y_true, y_pred, y_prob)

        if best_metrics is None:
            best_threshold = threshold
            best_metrics = metrics
            best_pred = y_pred
        else:
            old_key = (
                best_metrics["gmean"],
                best_metrics["recall"],
                best_metrics["accuracy"],
                best_metrics["specificity"],
                best_metrics["f1"]
            )
            new_key = (
                metrics["gmean"],
                metrics["recall"],
                metrics["accuracy"],
                metrics["specificity"],
                metrics["f1"]
            )

            if new_key > old_key:
                best_threshold = threshold
                best_metrics = metrics
                best_pred = y_pred

    return best_threshold, best_metrics, best_pred


# ============================================================
# 2. LogisticRegression 하이퍼파라미터 후보
# ============================================================

def make_logistic_l2_param_grid(
    C_grid=None,
    solver_grid=None,
    class_weight_grid=None,
    random_state=1
):
    """
    전진/후진/Stepwise/전역 선택법용 LogisticRegression 후보.
    변수선택은 wrapper가 담당하므로 penalty는 L2로 고정.
    """

    if C_grid is None:
        C_grid = [0.01, 0.1, 1, 10]

    if solver_grid is None:
        solver_grid = ["liblinear","lbfgs"]

    if class_weight_grid is None:
        # ADASYN 전이면 불균형 데이터이므로 balanced도 후보에 넣는 게 맞음
        class_weight_grid = ["balanced"]

    param_grid = []

    for C in C_grid:
        for solver in solver_grid:
            for class_weight in class_weight_grid:
                param_grid.append({
                    "penalty": "l2",
                    "C": C,
                    "solver": solver,
                    "class_weight": class_weight,
                    "max_iter": 10000,
                    "random_state": random_state
                })

    return param_grid


def build_logistic_pipeline(params, scaler_type="standard"):
    """
    scaler + LogisticRegression pipeline 생성.
    scaler_type:
    - "standard"
    - "minmax"
    - None
    """

    steps = []

    if scaler_type == "standard":
        steps.append(("scaler", StandardScaler()))
    elif scaler_type == "minmax":
        steps.append(("scaler", MinMaxScaler()))
    elif scaler_type is None:
        pass
    else:
        raise ValueError("scaler_type은 'standard', 'minmax', None 중 하나여야 함.")

    steps.append(("model", LogisticRegression(**params)))

    return Pipeline(steps)


# ============================================================
# 3. AIC/BIC, coef 계산
# ============================================================

def calculate_aic_bic(fitted_model, X, y, n_features):
    """
    LogisticRegression 기준 approximate AIC/BIC.
    규제 로지스틱에서 완전한 classical AIC/BIC는 아니고 비교용 지표로 사용.
    """
    proba = fitted_model.predict_proba(X)
    proba = np.clip(proba, 1e-15, 1 - 1e-15)

    y_arr = np.asarray(y).astype(int)
    ll = np.sum(
        y_arr * np.log(proba[:, 1]) +
        (1 - y_arr) * np.log(proba[:, 0])
    )

    n = len(y_arr)
    k = n_features + 1

    aic = 2 * k - 2 * ll
    bic = k * np.log(n) - 2 * ll

    return aic, bic


def get_coef_df(fitted_model, features):
    """
    pipeline 안의 LogisticRegression 계수 추출.
    """
    logit = fitted_model.named_steps["model"]
    coef = logit.coef_.ravel()

    coef_df = pd.DataFrame({
        "feature": features,
        "coef": coef,
        "abs_coef": np.abs(coef)
    }).sort_values("abs_coef", ascending=False).reset_index(drop=True)

    return coef_df


# ============================================================
# 4. 특정 변수 조합에 대해 Logistic 하이퍼파라미터 서칭
# ============================================================

def evaluate_feature_subset(
    X_train,
    y_train,
    X_valid,
    y_valid,
    features,
    param_grid=None,
    scaler_type="standard",
    threshold_search=True,
    threshold_grid=None,
    random_state=1
):
    """
    특정 feature subset에 대해:
    train 학습 → valid 검증 → LogisticRegression hyperparameter search.
    선택 기준은 valid G-Mean.
    """

    if param_grid is None:
        param_grid = make_logistic_l2_param_grid(random_state=random_state)

    X_tr = X_train[features]
    X_va = X_valid[features]

    records = []

    best_model = None
    best_params = None
    best_threshold = 0.5
    best_metrics = None

    for params in param_grid:
        try:
            model = build_logistic_pipeline(params, scaler_type=scaler_type)
            model.fit(X_tr, y_train)

            y_prob = model.predict_proba(X_va)[:, 1]

            if threshold_search:
                threshold, metrics, y_pred = find_best_threshold(
                    y_valid,
                    y_prob,
                    threshold_grid=threshold_grid
                )
            else:
                threshold = 0.5
                y_pred = model.predict(X_va)
                metrics = classification_metrics(y_valid, y_pred, y_prob)

            aic, bic = calculate_aic_bic(
                model,
                X_tr,
                y_train,
                n_features=len(features)
            )

            row = {
                "features": list(features),
                "n_features": len(features),
                "threshold": threshold,
                "C": params["C"],
                "solver": params["solver"],
                "class_weight": params["class_weight"],
                "valid_accuracy": metrics["accuracy"],
                "valid_precision": metrics["precision"],
                "valid_recall": metrics["recall"],
                "valid_specificity": metrics["specificity"],
                "valid_f1": metrics["f1"],
                "valid_gmean": metrics["gmean"],
                "valid_auc": metrics["auc"],
                "tn": metrics["tn"],
                "fp": metrics["fp"],
                "fn": metrics["fn"],
                "tp": metrics["tp"],
                "aic": aic,
                "bic": bic
            }

            records.append(row)

            if best_metrics is None:
                best_model = model
                best_params = params
                best_threshold = threshold
                best_metrics = metrics
            else:
                old_key = (
                    best_metrics["gmean"],
                    best_metrics["recall"],
                    best_metrics["accuracy"],
                    best_metrics["specificity"],
                    best_metrics["f1"]
                )
                new_key = (
                    metrics["gmean"],
                    metrics["recall"],
                    metrics["accuracy"],
                    metrics["specificity"],
                    metrics["f1"]
                )

                if new_key > old_key:
                    best_model = model
                    best_params = params
                    best_threshold = threshold
                    best_metrics = metrics

        except Exception as e:
            records.append({
                "features": list(features),
                "n_features": len(features),
                "C": params.get("C", None),
                "solver": params.get("solver", None),
                "class_weight": params.get("class_weight", None),
                "error": str(e)
            })

    result_df = pd.DataFrame(records)

    if "valid_gmean" in result_df.columns:
        result_df = result_df.dropna(subset=["valid_gmean"])
        result_df = result_df.sort_values(
            by=["valid_gmean", "valid_recall", "valid_accuracy", "aic", "bic"],
            ascending=[False, False, False, True, True]
        ).reset_index(drop=True)

    if best_model is None:
        raise RuntimeError("모든 LogisticRegression 후보가 실패함.")

    return {
        "model": best_model,
        "params": best_params,
        "threshold": best_threshold,
        "metrics": best_metrics,
        "result_df": result_df
    }


# ============================================================
# 5. 결과 출력 함수
# ============================================================

def print_selection_summary(result, title="Selection Result"):
    print("=" * 80)
    print(title)
    print("=" * 80)

    print(f"선택 변수 개수: {len(result['selected_features'])}")
    print("선택 변수:")
    print(result["selected_features"])

    print("-" * 80)
    print("Best LogisticRegression Params")
    print(result["best_params"])

    print("-" * 80)
    print(f"Best Threshold: {result['best_threshold']:.2f}")

    m = result["valid_metrics"]

    print("-" * 80)
    print("Validation Metrics")
    print(f"Accuracy   : {m['accuracy']:.4f}")
    print(f"Precision  : {m['precision']:.4f}")
    print(f"Recall     : {m['recall']:.4f}")
    print(f"Specificity: {m['specificity']:.4f}")
    print(f"F1         : {m['f1']:.4f}")
    print(f"G-Mean     : {m['gmean']:.4f}")
    print(f"AUC        : {m['auc']:.4f}" if not np.isnan(m["auc"]) else "AUC        : nan")

    print("-" * 80)
    print("Confusion Matrix")
    print(f"TN: {m['tn']}, FP: {m['fp']}")
    print(f"FN: {m['fn']}, TP: {m['tp']}")
    print("=" * 80)

## Lasso

L1 Logistic Regression 기반 변수선택

C=1/lambda (규제 강도)

C가 작을수록 강한 규제 -> 적은 변수 선택

C가 클수록 약한 규제 -> 많은 변수 선택

In [12]:
def select_lasso_features(
    X_train,
    y_train,
    X_valid,
    y_valid,
    C_grid=None,
    threshold_grid=None,
    cv_splits=5,
    coef_tol=1e-6,
    class_weight=None,
    max_iter=20000,
    random_state=1
):
    # 기본값 처리
    if C_grid is None:
        C_grid = [0.001, 0.01, 0.1, 1, 10]
    if threshold_grid is None:
        threshold_grid = np.arange(0.1, 0.91, 0.05)

    
    
    results = []

    for C in C_grid:
        model = LogisticRegression(
            penalty="l1",
            solver="saga",
            C=C,
            class_weight=class_weight,
            max_iter=max_iter,
            random_state=random_state,
            n_jobs=-1
        )
        model.fit(X_train, y_train)
        valid_proba = model.predict_proba(X_valid)[:, 1]

        for threshold in threshold_grid:
            valid_pred = (valid_proba >= threshold).astype(int)
            gmean = gmean_score(y_valid, valid_pred)
            results.append({
                "C": C,
                "threshold": threshold,
                "gmean": gmean
            })

    result_df = pd.DataFrame(results)
    best_row = result_df.loc[result_df["gmean"].idxmax()]
    best_C = best_row["C"]
    best_threshold = best_row["threshold"]

    # 최종 모델 학습
    final_model = LogisticRegression(
        penalty="l1",
        solver="saga",
        C=best_C,
        class_weight=class_weight,
        max_iter=max_iter,
        random_state=1,
        n_jobs=-1
    )
    final_model.fit(X_train, y_train)
    coef = final_model.coef_.ravel()
    selected_features = X_train.columns[np.abs(coef) > coef_tol].tolist()
    coef_df = pd.DataFrame({
        "feature": X_train.columns,
        "coef": coef,
        "abs_coef": np.abs(coef)
    }).sort_values("abs_coef", ascending=False)

    return {
        "selected_features": selected_features,
        "best_C": best_C,
        "best_threshold": best_threshold,
        "cv_result": result_df,
        "coef_df": coef_df,
        "model": final_model
    }
    
    
def calculate_aic_bic(fitted_model, X, y, n_features):
    """
    LogisticRegression 기준 approximate AIC/BIC.
    규제 로지스틱에서 완전한 classical AIC/BIC는 아니고 비교용 지표로 사용.
    """
    proba = fitted_model.predict_proba(X)
    proba = np.clip(proba, 1e-15, 1 - 1e-15)

    y_arr = np.asarray(y).astype(int)
    ll = np.sum(
        y_arr * np.log(proba[:, 1]) +
        (1 - y_arr) * np.log(proba[:, 0])
    )

    n = len(y_arr)
    k = n_features + 1

    aic = 2 * k - 2 * ll
    bic = k * np.log(n) - 2 * ll

    return aic, bic


def get_coef_df(fitted_model, features):
    """
    pipeline 안의 LogisticRegression 계수 추출.
    """
    logit = fitted_model.named_steps["model"]
    coef = logit.coef_.ravel()

    coef_df = pd.DataFrame({
        "feature": features,
        "coef": coef,
        "abs_coef": np.abs(coef)
    }).sort_values("abs_coef", ascending=False).reset_index(drop=True)

    return coef_df


C는 1/Lambda , LogisticRegression의 threshold, coef_total을 하이퍼 파라미터로 둠

In [13]:
C_grid = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30]

lasso_result = select_lasso_features(
    X_train,
    y_train,
    X_valid,
    y_valid,
    C_grid=C_grid,
    cv_splits=5
)

lasso_features = lasso_result["selected_features"]
best_C = lasso_result["best_C"]
lasso_cv_result = lasso_result["cv_result"]
lasso_coef_df = lasso_result["coef_df"]
lasso_model = lasso_result["model"]

print("Best C:", best_C)
display("Selected features:", lasso_features)

display(lasso_cv_result.sort_values("gmean", ascending=False).head(5))
display(lasso_coef_df)

Best C: 1.0


'Selected features:'

['KODEX 200_Premium',
 'KOSDAQ_return(%)',
 'NASDAQ_return(%)',
 'Brent Crude Oil_return(%)',
 'KOSPI 200 lagged_1_return(%)',
 'VKOSPI_Close',
 'VKOSPI_Change(%)',
 'KOSPI 200_DMI14',
 'KOSPI 200_ADX14',
 'KOSPI 200_OG',
 'Signal1_Buy',
 'Signal2_Buy',
 'Signal2_Sell',
 'Signal3_Buy',
 'Signal3_Sell',
 'Signal4_Buy',
 'Signal4_Sell']

,C,threshold,gmean
102,1.0,0.1,0.654299
85,0.3,0.1,0.642402
68,0.1,0.1,0.641980
119,3.0,0.1,0.641260
136,10.0,0.1,0.628089


,feature,coef,abs_coef
2,NASDAQ_return(%),-8.165476,8.165476
18,KOSPI 200_OG,-2.177822,2.177822
10,VKOSPI_Close,1.992498,1.992498
11,VKOSPI_Change(%),-1.648503,1.648503
13,KOSPI 200_DMI14,-1.240950,1.240950
8,KOSPI 200 lagged_1_return(%),-0.777298,0.777298
23,Signal2_Sell,-0.558452,0.558452
20,Signal1_Buy,0.462931,0.462931
3,Brent Crude Oil_return(%),-0.436470,0.436470
27,Signal4_Sell,-0.405458,0.405458


G-mean기준, tie breaker: AIC,BIC